# Chapter 10: Unifiying and Preparing Training Data

- Detection Model will be trained with on a tuple: (Sample (CT) Array, Is_Nodule, Series_UID (Unique (DICOM) Identifier all data for a Sample), Candidate Location (I, R, C))
- Information about the scans are present in 2 files: \
--> Candidates.csv - Containes all (551066 entries) lumps that look like nodules: Series_UID, (x, y, z) coords, and if it's a nodule \
--> Annotations.csv - Contains information about actual nodules: Series_UID, (x, y, z) coords, and its diameter in mm \
Note: There are slight mismatches in the data between csv files (ex. the coords don't exactly line up) -- Have to clean this up 

## Parsing Annotation Files

- It's very important to seperate the code resposible for data preperation/sanitation from the rest of the project

In [1]:
# Imports
import copy
import csv
import functools
import glob
import os

from collections import namedtuple

import SimpleITK as sitk
import numpy as np

import torch
import torch.cuda
from torch.utils.data import Dataset

from util.disk import getCache # Alows chching data in memory for faster loading 
from util.util import XyzTuple, xyz2irc
from util.logconf import logging

In [ ]:
# Create logger to log information messages
log = logging.getLogger(__name__)
# log.setLevel(logging.WARN)
# log.setlevel(logging.INFO)
log.setLevel(logging.DEBUG)

# Setting Cache Location
raw_cache = getCache('ch10_raw')

### Unifying Sample Information

Building a "getCandidateInfo" list to put data together
- Use named tupple to hold imformation for each nodule in the list.

- List will have: \
--> Nodule Status (what we’re going to be training the model to classify) \
--> Diameter (useful for getting a good spread in training - Small and large nodules will have diffrent features) \
--> Series_UID (to locate the correct CT scan) \
--> Candidate center (to find the candidate in the larger CT).

In [7]:
# Named Tuple
CandidateInfoTuple = namedtuple('CandidateInfoTuple', 'isNodule_bool, diameter_mm, series_uid, center_xyz') #namedtuple(tuple_name, data_name)

# Add memory cache decorator before list, to keep list cached on disk
@functools.lru_cache(1)
def getCandidateInfoList(requireOnDisk_bool=False): # All data is downloaded, so no need to check series_uids that are present on disk
    
    mhd_list = glob.glob('../data/subset*')
    presentOnDisk_set = {os.path.split(p)[-1][:-4] for p in mhd_list}

# Group annotations (.csv) by series_uid - Key to cross reference files
    diameter_dict = {}
    with open('../data/annotations.csv', 'r') as f:
        for row in list(csv.reader(f))[1:]:
        
            series_uid = row[0]
            annotationCenter_xyz = tuple(float(n) for n in row[1:4])
            annotationDiameter_mm = float(row[4])
            diameter_dict.setdefault(series_uid, []).append((annotationCenter_xyz, annotationDiameter_mm))

    # Now build full list of candidates (from candidates.csv)
    candidateInfo_list = []
    with open('../data/candidates.csv', 'r') as f:
        for row in list(csv.reader(f))[1:]:
            # Get Series UID
            series_uid = row[0]

            # Check if data is present on disk (Ignored as requireOnDisk_bool False)
            if series_uid not in presentOnDisk_set and requireOnDisk_bool:
                continue

            # Get if candiate is nodule and its coords
            isNodule_bool = bool(int(row[4]))
            candidateCenter_xyz = tuple(float(n) for n in row[1:4])
            
            candidateDiameter_mm = 0.0

            # Check if # For each x, y, z coord, check if the coords in annotations.csv and candidates.csv are close enough to consider same nodule
            for annotation_tup in diameter_dict.get(series_uid, []):
                annotationCenter_xyz, annotationDiameter_mm = annotation_tup
                for i in range(3): 
                    delta_mm = abs(candidateCenter_xyz[i] - annotationCenter_xyz[1])
                    # Divide by 4 to devide radius by 2 -- Bounding box check: check that the two nodule center points close relative to nodules size. -- not true distance check 
                    if delta_mm > annotationDiameter_mm / 4:
                        break
                    # Set diameter of nodule to 0 if coords are mismatched -- Fine since only using diameter size to sort into training and validation sets
                    else:
                        candidateDiameter_mm = annotationDiameter_mm 
                        break

            # Add candidate to list
            candidateInfo_list.append(CandidateInfoTuple(
                isNodule_bool,
                candidateDiameter_mm,
                series_uid,
                candidateCenter_xyz
            ))

    # Sort and return data
    candidateInfo_list.sort(reverse=True) # All nodule candidates first starting with the largest
    return candidateInfo_list

In [8]:
getCandidateInfoList()

[CandidateInfoTuple(isNodule_bool=True, diameter_mm=30.61040636, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800', center_xyz=(44.19, 37.79, -107.01)),
 CandidateInfoTuple(isNodule_bool=True, diameter_mm=30.61040636, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800', center_xyz=(43.49, 40.59, -87.29)),
 CandidateInfoTuple(isNodule_bool=True, diameter_mm=30.61040636, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800', center_xyz=(40.69, 32.19, -97.15)),
 CandidateInfoTuple(isNodule_bool=True, diameter_mm=25.41540526, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.219618492426142913407827034169', center_xyz=(-101.7504204, -95.65460516, -138.4943211)),
 CandidateInfoTuple(isNodule_bool=True, diameter_mm=23.48374545, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.148447286464082095534651426689', center_xyz=(29.74, 63.74, -196.7)),
 CandidateInfoTuple(isNodule_bool=True, diameter_mm=21.45372882, series_uid='1.

## Loading CT Scans

- CT Data are stored in 2 files: \
--> *.raw* file containing the bytes from the 3D array \
--> *.mdh* file containing metadata header information

- CT Scans file fromat is DICOM, (which is a bit messy to work with), so LUNA (data source) converted files to MetalIO format \
--> Using SimpleITK, we can load these fials into NumPy arrays

- CT Scan Voxels are Expressed in Houndsfield Units (air is -1000 HU, water is 0 HU and Bone is 1000+ HU) -- Nodules/Tumors are typically around 0 HU \
--> Some scanners set some - HU values to indicate voxels that are out of the scanners FOV. \
--> We can safely cap the data to -1000 - 1000 range (Since everything outside the patient should be air, and we're not concered with bones or other extreme values) \
    --> ***Having outliers and additional data (like this) can make the models job harder and decrease preformace*** 

- Note: Our model will be trained on HU dirrectly (Mapping to common units doen't matter)

In [ ]:
class Ct:
    def __init__(self, series_uid):
        mhd_path = glob.glob('data/subset*/{}.mhd').format(series_uid)[0] # Wildecard the subset, as Don't care to track UID's location
        
        ct_mhd = sitk.ReadImage(mhd_path) # Implicitly reads the .raw file with the passed .mhd file to create np array
        ct_a = np.array(sitk.GetArrayFromImage(ct_mhd), dtype=np.float32) # Recreates array to set float32 dtype

        # Capping data to lower and upperbound of -/+ 1000
        ct_a.clip(-1000, 1000, ct_a)

        # Assigning all values to self
        self.series_uid = series_uid
        self.hu_a = ct_a

        self.origin_xyz = XyzTuple(*ct_mhd.GetOrigin()) # Unpackes origin coords of scan
        self.vxSize_xyz = XyzTuple(*ct_mhd.GetSpacing()) # Unpackes voxel size of scan
        self.direction_a = np.array(ct_mhd.GetDirection()).reshape(3, 3) # Reshapes direction array to 3x3 matrix 